# 0. Problem
## 1907. Count Salary Categories — Medium
Count accounts in three fixed categories: `Low Salary` (< 20000), `Average Salary` (20000–50000), and `High Salary` (> 50000). Return all three categories even when a category count is zero.

Official: https://leetcode.com/problems/count-salary-categories/

# 1. Setup

In [ ]:
import pandas as pd
accounts_rows=[(3,108939),(2,12747),(8,87709),(6,91796)]
accounts_pd=pd.DataFrame(accounts_rows,columns=["account_id","income"])
accounts_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
accounts_spark=spark.createDataFrame(accounts_rows,["account_id","income"])
accounts_spark.createOrReplaceTempView("Accounts")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT 'Low Salary' AS category, SUM(CASE WHEN income<20000 THEN 1 ELSE 0 END) AS accounts_count FROM Accounts
UNION ALL
SELECT 'Average Salary', SUM(CASE WHEN income BETWEEN 20000 AND 50000 THEN 1 ELSE 0 END) FROM Accounts
UNION ALL
SELECT 'High Salary', SUM(CASE WHEN income>50000 THEN 1 ELSE 0 END) FROM Accounts
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
result_pd=pd.DataFrame({"category":["Low Salary","Average Salary","High Salary"],"accounts_count":[int(accounts_pd["income"].lt(20000).sum()),int(accounts_pd["income"].between(20000,50000,inclusive="both").sum()),int(accounts_pd["income"].gt(50000).sum())]})
result_pd

# 4. PySpark Solution

In [ ]:
result_spark=accounts_spark.agg(F.sum(F.when(F.col("income")<20000,1).otherwise(0)).alias("low"),F.sum(F.when(F.col("income").between(20000,50000),1).otherwise(0)).alias("average"),F.sum(F.when(F.col("income")>50000,1).otherwise(0)).alias("high")).select(F.explode(F.array(F.struct(F.lit("Low Salary").alias("category"),F.col("low").alias("accounts_count")),F.struct(F.lit("Average Salary").alias("category"),F.col("average").alias("accounts_count")),F.struct(F.lit("High Salary").alias("category"),F.col("high").alias("accounts_count")))).alias("x")).select("x.category","x.accounts_count")
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| fixed output categories | `UNION ALL` branches | construct result DataFrame | aggregate then reshape fixed structs |
| conditional count | `SUM(CASE...)` | boolean `.sum()` | `sum(when(...))` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Accounts

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: accounts_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: accounts_spark